### Adicionando a raiz do projeto no sistema

In [1]:
import sys
sys.path.append('/home/thehprogrammer/Projects/python/qa-method')

### PARSING

In [2]:
from backend.app.services.parser import ParserService
from backend.app.services.extract_content import ExtractContentService

/home/thehprogrammer/Projects/python/qa-method/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# PATH = "/home/thehprogrammer/Projects/python/qa-method/data/pdf/error/PEGC0702-T.pdf"
PATH = "/home/thehprogrammer/Projects/python/qa-method/data/pdf/error/PEGC0693-D.pdf"

In [4]:
parser_service = ParserService()
extract_content_service = ExtractContentService()

In [5]:
with open(PATH, 'rb') as file:
    pdf_content = file.read()

In [6]:
text = parser_service.extrair_texto(pdf_content)
print(text)

incorrect startxref pointer(1)


Erro ao extrair texto: '/Root'
Tentando reparar o PDF...
PDF reparado com sucesso.
  
 
UNIVERSIDADE FEDERAL DE SANTA CATARINA   
CENTRO TECNOLÓGICO  
PROGRAMA DE PÓS ­GRADUAÇÃO EM ENGENHARIA E GESTÃO DO 
CONHECIMENTO  
 
 
 
 
 
Jefferson de Oliveira Chaves  
 
 
 
 
 
 
EGCFlow: Uma aplicação de apoio ao ciclo de vida de dados  abertos conectados  
 
 
 
 
 
 
 
 
 
Florianópolis 
2021

 
Jefferson de Oliveira Chaves  
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
EGCFlow: Uma aplicação de apoio ao ciclo de vida de dados abertos conectados  
 
 
 
 
 
 
Dissertação submetida ao Programa de Pós ­Graduação 
em  Engenharia  e  Gestão  do  Conhecimento  da 
Universidade Federal de Santa Catarina para a obtenção 
do  título  de  Mestre  em  Engenharia  e  Gestão  do 
Conhecimento. 
Orientador: Prof. José Leomar Todesco, Dr.  
Coorientador: Prof. Alexandre Leopoldo Gonçalves, Dr.  
 
 
 
 
 
 
 
 
 
 
 
 
 
Florianópolis 
2021 
Ficha de identificação da obra elaborada pelo autor,
 através do Programa de G

### EXTRACT CONTENT WITH GEMINI-1.5-FLASH

In [11]:
resposta_llm = ''

In [12]:
response_chunks = extract_content_service.extrair_dados_llm(text)

In [13]:
for chunk in response_chunks:
    resposta_llm += chunk
    print(chunk)

```
json
{
    "autor": "Jefferson de Oliveira Chaves",
    
"titulo": "EGCFlow: Uma aplicação de apoio ao ciclo de vida
 de dados abertos conectados",
    "tipo": "Dissertação",
    "area_de_concentracao": "Engenharia do
 Conhecimento",
    "ano_de_publicacao": "2021",
    "local": "Florianópolis",
    "orienta
dor": "Prof. José Leomar Todesco, Dr.",
    "coorientador": "Prof. Alexandre Leopoldo Gonçalves, Dr.",
    "resumo": "A web se consolidou como principal meio para se publicar,
 obter e compartilhar informações na internet. Como uma expansão natural da web, surgiram os conceitos de Dados Abertos e Dados Abertos Conectados, que podem ser compreendidos como um conjunto de melhores práticas para publicação e
 conexão de conjuntos de dados de forma estruturada na web. Entretanto, o processo de produção e manutenção de conjuntos de dados abertos conectados, constitui-se como uma tarefa complexa e onerosa, mas necessária no cenário de expansão da web de dados. Diante disso, utilizando

In [14]:
print(resposta_llm)

```json
{
    "autor": "Jefferson de Oliveira Chaves",
    "titulo": "EGCFlow: Uma aplicação de apoio ao ciclo de vida de dados abertos conectados",
    "tipo": "Dissertação",
    "area_de_concentracao": "Engenharia do Conhecimento",
    "ano_de_publicacao": "2021",
    "local": "Florianópolis",
    "orientador": "Prof. José Leomar Todesco, Dr.",
    "coorientador": "Prof. Alexandre Leopoldo Gonçalves, Dr.",
    "resumo": "A web se consolidou como principal meio para se publicar, obter e compartilhar informações na internet. Como uma expansão natural da web, surgiram os conceitos de Dados Abertos e Dados Abertos Conectados, que podem ser compreendidos como um conjunto de melhores práticas para publicação e conexão de conjuntos de dados de forma estruturada na web. Entretanto, o processo de produção e manutenção de conjuntos de dados abertos conectados, constitui-se como uma tarefa complexa e onerosa, mas necessária no cenário de expansão da web de dados. Diante disso, utilizando-se o m

### SAVE the JSON (STANDARD)

In [15]:
import json
import os
import re

In [16]:
json_padrao = {
    "autor": "",
    "titulo": "",
    "tipo": "",
    "area_de_concentracao": "",
    "ano_de_publicacao": "",
    "local": "",
    "orientador": "",
    "coorientador": "",
    "resumo": "",
    "palavras_chave": [],
    "abstract": "",
    "keywords": [],
    "introducao": {
        "contextualizacao": "",
        "problematica": "",
        "ineditismo": "",
        "contribuição": ""
    },
    "conclusao": ""
}

In [17]:
def sanitizar_json(resposta_llm: str) -> str:
    # Remove duplicações de `}}` ou `} \n }` no final do JSON
    resposta_llm = re.sub(r'}\s*}', '}', resposta_llm)
    
    # Remove qualquer texto após o fechamento da última chave }
    resposta_llm = resposta_llm.strip()

    # Verifica se existe conteúdo além do JSON, removendo tudo depois do último "}"
    ultima_chave = resposta_llm.rfind("}")
    if ultima_chave != -1:
        resposta_llm = resposta_llm[:ultima_chave+1]
    
    return resposta_llm

In [18]:
# Função para padronizar o JSON, removendo chaves extras e preenchendo as chaves corretas
def padronizar_conteudo(content: dict, padrao: dict) -> dict:
    def padronizar(d, padrao):
        if isinstance(padrao, dict):
            # Se for um dicionário, verifica as sub-chaves
            return {k: padronizar(d.get(k, v), v) for k, v in padrao.items()}
        # Se for um valor simples, mantém o valor se existir ou preenche com o padrão
        return d if d is not None else padrao

    # Padroniza o conteúdo
    padronizado = padronizar(content, padrao)

    # Verifica se o tipo é dissertação, e se for, limpa 'ineditismo' e 'contribuição'
    if padronizado.get("tipo") == "dissertação" or padronizado.get("tipo") == "Dissertação":
        padronizado["introducao"]["ineditismo"] = ""
        padronizado["introducao"]["contribuição"] = ""

    return padronizado


In [19]:
# Função para remover chaves que não estão no padrão
def remover_chaves_extras(content: dict, padrao: dict) -> dict:
    if isinstance(content, dict):
        # Remove chaves que não estão no padrão
        return {k: remover_chaves_extras(content.get(k, padrao.get(k)), padrao[k]) for k in padrao if k in content}
    return content

In [20]:
# Função para salvar o conteúdo em um arquivo JSON
def salvar_json(content: dict, output_path: str) -> None:
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as file:
        json.dump(content, file, indent=4, ensure_ascii=False)

In [21]:
# Função para processar a resposta do LLM e salvar no arquivo JSON
def processar_e_salvar_resposta_llm(resposta_llm: str, output_path: str):
    # Sanitiza o JSON para corrigir possíveis erros de formatação
    resposta_llm_sanitizada = sanitizar_json(resposta_llm)
    
    try:
        # Converte a string JSON sanitizada da LLM em um dicionário
        content_gerado = json.loads(resposta_llm_sanitizada)
    except json.JSONDecodeError as e:
        print(f"Erro ao decodificar JSON: {e}")
        return

    # Remove chaves extras que não estão no padrão
    content_sem_extras = remover_chaves_extras(content_gerado, json_padrao)
    
    # Preenche o padrão com os valores gerados
    content_padronizado = padronizar_conteudo(content_sem_extras, json_padrao)

    # Salva o conteúdo padronizado em um arquivo JSON
    salvar_json(content_padronizado, output_path)
    print(f"Conteúdo salvo em {output_path}")



In [22]:
OUTH_PATH = '/home/thehprogrammer/Projects/python/qa-method/test/services/llm_output.json'

In [23]:
# Removendo espaços em branco e formatação adicional
resposta_llm = resposta_llm.strip()

# Removendo backticks se eles estiverem presentes
resposta_llm = resposta_llm.replace('```json', '').replace('```', '')

In [24]:
processar_e_salvar_resposta_llm(resposta_llm, OUTH_PATH)

Conteúdo salvo em /home/thehprogrammer/Projects/python/qa-method/test/services/llm_output.json


### Create Service

O objetivo é criar um microserviço para extrair o conteúdo de documentos de teses ou dissertações em PDF, permitindo que o usuário customize as chaves e regras de extração e visualize o resultado em tempo real. Vamos analisar os componentes sugeridos e fornecer sugestões de melhoria.
Componentes do Serviço:

- Upload de Documento PDF:
    - O usuário faz o upload do PDF. A análise do documento é feita no backend usando técnicas de processamento de linguagem natural (NLP).
    - Sugestão: O backend pode incluir uma verificação preliminar do tipo do documento (tese ou dissertação) antes de iniciar o processamento para garantir que o documento segue as especificações.

- Padrão Sugerido em Formato JSON:
    - Exibir um padrão de roteiro no formato JSON. O usuário pode ajustar as chaves conforme a estrutura do documento.
    - Sugestão: Limitar o tipo de ajuste que o usuário pode fazer. Certas chaves obrigatórias podem ser protegidas para evitar erros de formatação. Além disso, permitir a validação do novo JSON modificado antes de aplicar a extração.

- Regras de Extração Personalizáveis:
    - O usuário pode modificar as regras para as chaves (por exemplo, o que deve ser extraído para "resumo", "abstract", etc.).
    - Sugestão: Incluir descrições breves para cada regra e exemplos práticos para guiar o usuário. Isso ajudaria a evitar modificações inadequadas que poderiam gerar erros de extração.

- Exibição em Tempo Real da Resposta Gerada:
    - Enquanto o sistema processa o PDF, a resposta seria exibida progressivamente ao usuário.
    - Sugestão: Adicionar um sistema de notificação ou barra de progresso, informando o usuário sobre o andamento do processamento. Isso é especialmente importante se o documento for grande.

- Botão para Salvar o JSON Final:
    - Após visualizar a resposta gerada, o usuário pode salvar o conteúdo extraído em formato JSON.
    - Sugestão: Permitir que o usuário baixe o JSON diretamente ou envie por e-mail. Além disso, pode ser útil incluir a opção de exportar o JSON em outros formatos, como CSV ou XML.